<a href="https://colab.research.google.com/github/Thcastro2004/ECSE551-A2-ML-for-engineers/blob/main/Barnett_Cottereau_Zhang_Assignment2_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import KFold
from PIL import Image
import os
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi
from tqdm import tqdm
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy
import json
from torch.cuda.amp import autocast, GradScaler


## Dataset Classes

In [ ]:
class Task2TrainingDataset224(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/train_labels.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/train/'
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
        self.transform = transforms.Compose([
            transforms.Resize(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.trainpath + self.df.at[idx, "id"]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        label_str = self.df.at[idx, "label"]
        label = self.labels_dict[label_str]
        return img, label

In [ ]:
class Task2TestDataset224(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/train_labels.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/train/'
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
        self.transform = transforms.Compose([
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.trainpath + self.df.at[idx, "id"]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        label_str = self.df.at[idx, "label"]
        label = self.labels_dict[label_str]
        return img, label

In [ ]:
class Test551(Dataset):
    def __init__(self, data_path='/content/drive/MyDrive/data'):
        super().__init__()
        self.df = pd.read_csv(f'{data_path}/kaggle_challenge/sample_submission.csv')
        self.trainpath = f'{data_path}/kaggle_challenge/test/'
        self.transform = transforms.Compose([
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.labels_dict = {
            "truck": 0, "deer": 1, "bird": 2, "frog": 3, "ship": 4,
            "horse": 5, "cat": 6, "dog": 7, "automobile": 8, "airplane": 9
        }
    
    def __getitem__(self, idx):
        img = Image.open(self.trainpath+self.df.at[idx, "id"])
        img = img.convert("RGB")
        img = self.transform(img)
        label = 0
        return img, label
    
    def __len__(self):
        return len(self.df)

In [ ]:
def create_efficientnet_b0(num_classes=10):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

## Task 2 Training with Cross-Validation

In [ ]:
# ================================================
# Task 2 Training with Cross-Validation (Colab)
# ================================================
from torch.cuda.amp import autocast, GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print("Device:", device)

# Datasets (10k labeled train)
train_dataset = Task2TrainingDataset224()
test_dataset  = Task2TestDataset224()   # same images, no augmentation

n_samples = len(train_dataset)
all_indices = np.arange(n_samples)
print("Total samples:", n_samples)

# Cross-validation config
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
num_epochs = 50
batch_size = 64
val_frequency = 2
num_workers = 2   # Colab: 2 workers is usually safe

fold_results = []
best_overall_acc = 0.0
best_model_state = None


def train_one_fold(train_idx, val_idx):
    """
    Trains EfficientNet-B0 on one CV fold.
    Returns: (best_val_acc, best_model_state_dict)
    """
    # Subsets
    train_subset = Subset(train_dataset, train_idx)
    val_subset   = Subset(test_dataset,  val_idx)

    # DataLoaders
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    # Model, loss, optimizer, scheduler, AMP
    model = create_efficientnet_b0(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = GradScaler()

    best_val_acc = 0.0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        # ---- Validation ----
        if (epoch + 1) % val_frequency == 0 or epoch + 1 == num_epochs:
            model.eval()
            correct = 0
            total = 0

            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs = imgs.to(device)
                    labels = labels.to(device)

                    outputs = model(imgs)
                    _, preds = outputs.max(1)
                    total += labels.size(0)
                    correct += (preds == labels).sum().item()

            val_acc = correct / total
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state = copy.deepcopy(model.state_dict())

            print(
                f"Epoch {epoch+1}/{num_epochs} "
                f"| Loss={running_loss:.4f} "
                f"| Val Acc={val_acc:.4f} "
                f"| Best={best_val_acc:.4f}"
            )

    return best_val_acc, best_state


print("Starting 5-fold cross validation...\n")

for fold, (train_idx, val_idx) in enumerate(kfold.split(all_indices)):
    print(f"\n===== Fold {fold+1} =====")
    best_acc, model_state = train_one_fold(train_idx, val_idx)
    fold_results.append(best_acc)

    if best_acc > best_overall_acc:
        best_overall_acc = best_acc
        best_model_state = model_state
        torch.save(best_model_state, "/content/drive/MyDrive/task2_best_model.pth")
        print(f"Best model saved from Fold {fold+1}!")

print("\nResults per fold:", fold_results)
print("Mean CV accuracy:", np.mean(fold_results))
print("Best CV accuracy:", best_overall_acc)


## Task 2 Results Saving

In [ ]:
# ======================================================
# Build Kaggle submission from best model (2000 test imgs)
# ======================================================
kaggle_test_dataset = Test551()  # uses data_path default
kaggle_test_loader = DataLoader(
    kaggle_test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_task2 = create_efficientnet_b0(num_classes=10).to(device)
model_task2.load_state_dict(torch.load('/content/drive/MyDrive/task2_best_model.pth'))
model_task2.eval()

idx_to_label = {
    0: "truck", 1: "deer", 2: "bird", 3: "frog", 4: "ship",
    5: "horse", 6: "cat", 7: "dog", 8: "automobile", 9: "airplane"
}

all_preds = []
all_ids = []

with torch.no_grad():
    for imgs, ids in kaggle_test_loader:
        imgs = imgs.to(device)
        outputs = model_task2(imgs)
        _, preds = outputs.max(1)
        preds = preds.cpu().numpy()

        all_ids.extend(ids)
        all_preds.extend([idx_to_label[int(p)] for p in preds])

submission_df = pd.DataFrame({"id": all_ids, "label": all_preds})
submission_df = submission_df.sort_values("id")
submission_df.to_csv("/content/drive/MyDrive/task2_kaggle_submission.csv", index=False)

print("Saved:", "/content/drive/MyDrive/task2_kaggle_submission.csv")


## Task 2 Kaggle Submission Generation

In [ ]:
# Load the test dataset for Kaggle submission
kaggle_test_dataset = Test551()
kaggle_test_loader = DataLoader(kaggle_test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Load best model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_task2 = create_efficientnet_b0(num_classes=10)
model_task2.load_state_dict(torch.load('/content/drive/MyDrive/task2_best_model.pth'))
model_task2 = model_task2.to(device)
model_task2.eval()

# Reverse label dictionary for converting predictions to labels
idx_to_label = {0: "truck", 1: "deer", 2: "bird", 3: "frog", 4: "ship",
                5: "horse", 6: "cat", 7: "dog", 8: "automobile", 9: "airplane"}

# Generate predictions
predictions = []
ids = []

with torch.no_grad():
    for inputs, _ in tqdm(kaggle_test_loader, desc='Generating predictions'):
        inputs = inputs.to(device)
        outputs = model_task2(inputs)
        _, predicted = torch.max(outputs.data, 1)
        predictions.extend(predicted.cpu().numpy())

# Get IDs from the dataset
for idx in range(len(kaggle_test_dataset)):
    ids.append(kaggle_test_dataset.df.at[idx, 'id'])

# Convert predictions to labels
labels = [idx_to_label[pred] for pred in predictions]

# Create submission DataFrame
submission_df = pd.DataFrame({'id': ids, 'label': labels})

# Save submission file
submission_path = '/content/drive/MyDrive/task2_submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path}")
print(f"Total predictions: {len(predictions)}")